# L10b: GloVe Word Embeddings
In this lab, we train GloVe (Global Vectors for Word Representation) embeddings from scratch on a toy corpus. We build a distance-weighted co-occurrence matrix, optimize the GloVe objective using gradient descent, and compare the resulting embeddings to those from CBOW and Skip-Gram.

> __Learning Objectives__
> * __Build a distance-weighted co-occurrence matrix:__ Construct a word co-occurrence matrix that weights nearby word pairs more heavily than distant ones, and compare it to the binary co-occurrence matrix used in PMI.
> * __Train GloVe using weighted least-squares:__ Optimize the GloVe objective over non-zero co-occurrence entries and examine how the weighting function shapes the loss.
> * __Compare GloVe embeddings to CBOW and Skip-Gram:__ Use cosine similarity to evaluate whether GloVe captures word associations differently than the prediction-based models trained in the previous labs.

___

## Setup, Data, and Prerequisites

> We load packages and source files via `Include.jl`. This file loads `VLDataScienceMachineLearningPackage.jl` along with local implementations of CBOW, Skip-Gram, and GloVe used for comparison throughout the lab.

In [ ]:
include("Include.jl")

### Implementations

| Function | Location | Description |
|----------|----------|-------------|
| `build_weighted_cooccurrence_matrix(sentences, vocabulary; window_size)` | `src/GloVe.jl` | Builds co-occurrence matrix with $1/d$ distance weighting |
| `glove_weight(x; x_max, alpha)` | `src/GloVe.jl` | Capping function $f(x) = \min(1, (x/x_{\max})^\alpha)$ |
| `train_glove(X, vocab_size; d, eta, num_epochs, x_max, alpha)` | `src/GloVe.jl` | Trains GloVe via gradient descent over non-zero pairs |
| `train_cbow(training_pairs, vocab_size; d_h, eta, num_epochs)` | `src/CBOW.jl` | Trains CBOW for embedding comparison |
| `train_skipgram_ns(training_pairs, vocab_size, noise_dist; ...)` | `src/NegativeSampling.jl` | Trains Skip-Gram with negative sampling for comparison |

### Constants

In [ ]:
sentences = [
    "I love julia and machine learning",
    "julia is fun and great",
    "I enjoy data science and machine learning",
    "data science is great and fun",
    "machine learning is a great and fun subject",
    "I love learning new things about data science"
];

control_tokens = ["<bos>", "<eos>", "<pad>", "<unk>"];

window_size = 2;    # context window half-width
d = 5;              # GloVe embedding dimension
eta_glove = 0.05;   # GloVe learning rate
num_epochs = 500;   # training epochs
x_max = 100.0;      # co-occurrence count saturation threshold
alpha = 0.75;       # exponent for weighting function

### Data

In [ ]:
all_words = vcat([split(lowercase(s)) for s in sentences]...)
unique_words = unique(all_words)
vocabulary = Dict{String, Int64}()
for (i, token) in enumerate(control_tokens)
    vocabulary[token] = i
end
offset = length(control_tokens)
for (i, word) in enumerate(unique_words)
    if !haskey(vocabulary, word)
        vocabulary[word] = offset + i
    end
end
inverse_vocabulary = Dict{Int64, String}(v => k for (k, v) in vocabulary)
println("Vocabulary size: $(length(vocabulary))")

___

## Task 1: Build the Distance-Weighted Co-occurrence Matrix

> GloVe uses a distance-weighted co-occurrence matrix $\mathbf{X}$, where entry $X_{ij}$ accumulates $1/|i-j|$ for each occurrence of word $j$ within the window of word $i$. Words that are closer together contribute more weight than distant words. This differs from the binary co-occurrence matrix used in PMI, which counts each co-occurrence equally regardless of distance.

In [ ]:
X = build_weighted_cooccurrence_matrix(sentences, vocabulary; window_size=window_size);
println("Co-occurrence matrix size: $(size(X))")
println("Number of non-zero entries: $(sum(X .> 0))")

We inspect specific entries of the co-occurrence matrix for selected word pairs.

In [ ]:
word_pairs = [
    ("machine", "learning"),
    ("data", "science"),
    ("julia", "fun"),
    ("love", "julia"),
    ("great", "fun"),
    ("machine", "julia"),
]

header = ["Word 1", "Word 2", "Weighted Co-occ X_ij"]
rows = []
for (w1, w2) in word_pairs
    i = get(vocabulary, w1, 0)
    j = get(vocabulary, w2, 0)
    if i > 0 && j > 0
        push!(rows, [w1, w2, round(X[i,j], digits=3)])
    end
end
pretty_table(hcat(rows...)'; header=header)

__Discussion:__ Compare the weighted co-occurrence values to the window-based co-occurrence counts from the PMI lab. Which pairs show the largest difference between the two representations? Why does distance weighting reduce the value for some pairs that appear in the same sentence?

___

## Task 2: Train GloVe Embeddings

> GloVe minimizes a weighted least-squares objective over all non-zero co-occurrence pairs:
> $$\mathcal{L} = \sum_{i,j:\,X_{ij}>0} f(X_{ij})\!\left(\mathbf{w}_i^{\top}\tilde{\mathbf{w}}_j + b_i + \tilde{b}_j - \log X_{ij}\right)^2$$
> where $\mathbf{w}_i \in \mathbb{R}^d$ are word vectors, $\tilde{\mathbf{w}}_j \in \mathbb{R}^d$ are context vectors, $b_i$ and $\tilde{b}_j$ are scalar biases, and $f(x) = \min(1, (x/x_{\max})^\alpha)$ caps the contribution of very frequent co-occurrence pairs. The model is trained with gradient descent over the four parameter groups $\{\mathbf{W}, \tilde{\mathbf{W}}, \mathbf{b}, \tilde{\mathbf{b}}\}$.

In [ ]:
Random.seed!(42)
W_word, W_ctx, b_word, b_ctx, loss_glove = train_glove(X, length(vocabulary);
    d=d, eta=eta_glove, num_epochs=num_epochs, x_max=x_max, alpha=alpha);

plot(loss_glove; xlabel="Epoch", ylabel="Average Weighted Squared Loss",
    title="GloVe Training Loss", legend=false, lw=2)

__Discussion:__ GloVe only optimizes over non-zero co-occurrence pairs. On this small corpus, how many non-zero pairs are there compared to the full $|V|^2$ pairs? How would training efficiency scale on a large corpus where most word pairs never co-occur?

___

## Task 3: Extract Combined Embeddings and Analyze Similarities

> The final GloVe embedding for word $i$ is the sum of its word vector and context vector: $\mathbf{e}_i = \mathbf{w}_i + \tilde{\mathbf{w}}_i$. This combination uses information from both the word and context roles that each word plays in the corpus.

In [ ]:
embeddings_glove = W_word .+ W_ctx;
println("GloVe embedding matrix size: $(size(embeddings_glove))  (d × vocab_size)")

In [ ]:
word_pairs_sim = [
    ("machine", "learning"),
    ("data", "science"),
    ("julia", "fun"),
    ("love", "enjoy"),
    ("great", "fun"),
    ("machine", "julia"),
]

cosine_sim_fn(E, w1, w2, vocab) = begin
    i = get(vocab, w1, 0); j = get(vocab, w2, 0)
    (i == 0 || j == 0) && return NaN
    e1 = E[:, i]; e2 = E[:, j]
    dot(e1, e2) / (norm(e1) * norm(e2) + 1e-10)
end

header = ["Word 1", "Word 2", "GloVe Cosine Sim."]
rows = []
for (w1, w2) in word_pairs_sim
    sim = round(cosine_sim_fn(embeddings_glove, w1, w2, vocabulary), digits=3)
    push!(rows, [w1, w2, sim])
end
pretty_table(hcat(rows...)'; header=header)

___

## Task 4: Compare GloVe to CBOW and Skip-Gram

> We train CBOW and Skip-Gram (with negative sampling) on the same corpus and compare cosine similarities across all three models. GloVe optimizes over global co-occurrence statistics, while CBOW and Skip-Gram are trained through local prediction tasks. The theoretical connection between the methods is established by Levy and Goldberg (2014), who showed that Skip-Gram with negative sampling implicitly factorizes a shifted PMI matrix — the same statistical structure that GloVe makes explicit.

In [ ]:
# train CBOW
cbow_pairs = build_cbow_pairs(sentences, vocabulary; window_size=window_size)
Random.seed!(42)
W1_cbow, _, _ = train_cbow(cbow_pairs, length(vocabulary); d_h=d, eta=0.01, num_epochs=num_epochs)

# train Skip-Gram with Negative Sampling
sg_pairs = build_skipgram_pairs(sentences, vocabulary; window_size=window_size)
noise_dist = build_noise_distribution(sentences, vocabulary)
Random.seed!(42)
W1_ns, _, _ = train_skipgram_ns(sg_pairs, length(vocabulary), noise_dist;
    d_h=d, eta=0.01, num_epochs=num_epochs, k=5)

In [ ]:
header = ["Word 1", "Word 2", "GloVe", "CBOW", "SG Neg. Samp."]
rows = []
for (w1, w2) in word_pairs_sim
    g  = round(cosine_sim_fn(embeddings_glove, w1, w2, vocabulary), digits=3)
    c  = round(cosine_sim_fn(W1_cbow,          w1, w2, vocabulary), digits=3)
    ns = round(cosine_sim_fn(W1_ns,            w1, w2, vocabulary), digits=3)
    push!(rows, [w1, w2, g, c, ns])
end
pretty_table(hcat(rows...)'; header=header)

__Discussion:__ GloVe makes global co-occurrence statistics explicit while Skip-Gram implicitly factorizes a shifted PMI matrix. On this small corpus, do the three models produce consistent similarity rankings? Which model appears most sensitive to corpus size? How would you expect results to differ on a corpus with millions of sentences?

___

## Summary

In this lab, we trained GloVe embeddings by building a distance-weighted co-occurrence matrix and minimizing a weighted least-squares objective over non-zero pairs. We then compared the resulting embeddings to CBOW and Skip-Gram, showing that all three methods capture word associations from co-occurrence statistics, but through different training objectives.

> __Key Takeaways__
> * **GloVe uses global co-occurrence statistics:** GloVe builds a distance-weighted co-occurrence matrix and minimizes a weighted least-squares objective over all non-zero pairs, making the global structure of the corpus explicit in the training objective.
> * **The weighting function controls the contribution of frequent pairs:** The capping function $f(x) = \min(1, (x/x_{\max})^\alpha)$ prevents high-frequency co-occurrence pairs from dominating the loss, improving embedding quality for rare but informative word pairs.
> * **GloVe, CBOW, and Skip-Gram are connected through PMI factorization:** Levy and Goldberg (2014) showed that Skip-Gram with negative sampling implicitly factorizes a shifted PMI matrix, the same statistical structure that GloVe optimizes directly — linking the three approaches through the underlying co-occurrence structure of the corpus.